In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sklearn.cluster import KMeans
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
    
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from sklearn.preprocessing import PowerTransformer

In [2]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [3]:
# Need to choose patient_id from OUS_D1 in response_OUS
data = list(OUS_D1['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D2 with response_OUS
clinical_train = pd.merge(OUS_D1, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS', 'LRC', 'event_LRC'])]

In [4]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [5]:
# Check null values in D2 
clinical_train.isnull().sum().sum()

0

In [6]:
""" Takes too long time 
# Check collinearity for OUS data 
df = clinical_train

# Create a correlation matrix
correlation_matrix = df.corr()

plt.figure(figsize=(15, 12))

# Plot a heatmap
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.show()
""" 

" Takes too long time \n# Check collinearity for OUS data \ndf = clinical_train\n\n# Create a correlation matrix\ncorrelation_matrix = df.corr()\n\nplt.figure(figsize=(15, 12))\n\n# Plot a heatmap\nsns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)\nplt.show()\n"

## Test dataset: MAASTRO 

In [7]:
(MAASTRO_D1['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [8]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [9]:
# need to choose patient_id from MAASTRO_D2 in response_MAASTRO
data = list(MAASTRO_D1['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [10]:
# Merge MAASTRO_D1 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D1, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'OS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG,DFS,DFS_event
0,1,55,0,0,1,0,0,1,1,1,0,0,15.438583,22.841,263.611623,62.43,0.0
1,2,55,0,0,1,0,0,0,0,0,20,1,8.829353,5.660,36.980700,60.00,0.0
2,3,55,0,0,1,0,0,0,0,1,6,1,13.476123,7.791,74.636342,8.83,1.0
3,4,61,1,0,0,0,1,1,0,1,45,1,8.632732,7.908,46.791979,19.73,1.0
4,6,70,0,0,1,0,0,1,1,1,59,0,9.783954,15.237,107.637514,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,66,1,0,0,0,1,0,0,0,55,1,31.338410,6.110,144.490782,13.27,1.0
95,111,63,0,0,0,0,1,0,0,1,174,1,13.041604,7.182,69.214868,85.87,1.0
96,112,63,0,0,1,0,0,1,1,1,0,1,8.944517,16.483,102.594274,42.87,0.0
97,113,54,0,0,1,0,0,1,1,0,0,0,14.184236,9.981,103.229492,58.93,0.0


In [11]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [12]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG,DFS,DFS_event


In [13]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS'])]

# y 
y = clinical_train.loc[:, ['DFS', 'event_DFS']]

# Set lower, upper time point and times for IBS calculation later 
lower, upper = np.percentile(y['DFS'], [10, 90])
times = np.arange(lower, upper)

In [14]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [15]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 14)
y_train:  (139,)


In [16]:
clinical_test.rename(columns = {'DFS_event' : 'event_DFS'}, inplace = True)

In [17]:
# X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'event_DFS'])]

# y y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['DFS', 'event_DFS']]
lower, upper = np.percentile(y_MAASTRO['DFS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_DFS'], y_MAASTRO['DFS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 16)

# Feature Selection: RENT

In [18]:
# Choose features from the result of RENT 
selected_features = ["hpv_related",
"pack_years",
"uicc8_III-IV",
"oropharynx",
"cavum_oris",
"TLG"]

In [19]:
X_rent = X.loc[:, selected_features]
X_new = X_rent.copy()

In [20]:
list(X_new)

['hpv_related',
 'pack_years',
 'uicc8_III-IV',
 'oropharynx',
 'cavum_oris',
 'TLG']

In [21]:
# Selecct the columns from X_MAASTRO
MAASTRO_new = X_MAASTRO.loc[:, selected_features]

# Yeo-Johnson Transformation

In [22]:
# Copy the original X for later 
original_X = X.copy()

In [28]:
# Transform X_new 
# Set the categorical_columns
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)

# Apply Yeo-Johnson transformation
pt = PowerTransformer(method='yeo-johnson')
X_new_numeric_transformed = pt.fit_transform(X_new_numeric)

# Create DataFrame with transformed numerical data
X_new_numeric_transformed = pd.DataFrame(X_new_numeric_transformed, 
                                         columns=X_new_numeric.columns, 
                                         index=X_new.index)

# Concatenate transformed numerical data with categorical data
X_new_std = pd.concat([X_new_numeric_transformed, X_new_categoric], axis=1)
X_new_std = X_new_std[X_new.columns]

# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the transformation for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = pt.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
X_new_std = X_new_std[X_new.columns]
MAASTRO_new = MAASTRO_new[X_new.columns]
MAASTRO_new_std = MAASTRO_new_std[X_new.columns]

In [29]:
X_new

,hpv_related,pack_years,uicc8_III-IV,oropharynx,cavum_oris,TLG
0,0.0,0.000000,0.0,1,0,86.228420
1,0.0,27.404795,0.0,0,0,7.040100
2,0.0,41.019178,1.0,0,1,83.569669
3,0.0,37.500000,0.0,0,0,5.567091
4,0.0,53.000000,0.0,0,0,16.150550
...,...,...,...,...,...,...
134,1.0,0.000000,0.0,1,0,26.280140
135,1.0,0.000000,1.0,1,0,101.754834
136,1.0,39.498630,0.0,1,0,66.273201
137,1.0,71.527397,1.0,1,0,71.832443


In [30]:
X_new_std

,hpv_related,pack_years,uicc8_III-IV,oropharynx,cavum_oris,TLG
0,0.0,-1.533345,0.0,1,0,0.343545
1,0.0,0.397233,0.0,0,0,-1.657833
2,0.0,0.830437,1.0,0,1,0.318631
3,0.0,0.727938,0.0,0,0,-1.835650
4,0.0,1.144253,0.0,0,0,-1.003396
...,...,...,...,...,...,...
134,1.0,-1.533345,0.0,1,0,-0.611074
135,1.0,-1.533345,1.0,1,0,0.474941
136,1.0,0.786825,0.0,1,0,0.133612
137,1.0,1.554125,1.0,1,0,0.197985


In [31]:
MAASTRO_new 

,hpv_related,pack_years,uicc8_III-IV,oropharynx,cavum_oris,TLG
0,1,0,0,1,0,263.611623
1,0,20,1,1,0,36.980700
2,0,6,1,1,0,74.636342
3,0,45,1,0,0,46.791979
4,1,59,0,1,0,107.637514
...,...,...,...,...,...,...
94,0,55,1,0,0,144.490782
95,0,174,1,0,0,69.214868
96,1,0,1,1,0,102.594274
97,1,0,0,1,0,103.229492


In [32]:
MAASTRO_new_std

,hpv_related,pack_years,uicc8_III-IV,oropharynx,cavum_oris,TLG
0,1,-1.533345,0,1,0,1.218887
1,0,0.104884,1,1,0,-0.335276
2,0,-0.713198,1,1,0,0.228548
3,0,0.940197,1,0,0,-0.145668
4,1,1.285347,0,1,0,0.519421
...,...,...,...,...,...,...
94,0,1.192311,1,0,0,0.751363
95,0,3.094727,1,0,0,0.168333
96,1,-1.533345,1,1,0,0.481447
97,1,-1.533345,0,1,0,0.486335


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [33]:
# Setting the y format for skf below  
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold( n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-13 17:35:37,492] A new study created in memory with name: no-name-c21d4a16-d53e-47e9-b572-13ba4b06217d


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6055776892430279
Fold 2 C-index: 0.5310077519379846
Fold 3 C-index: 0.7404255319148936
Fold 4 C-index: 0.6806083650190115


[I 2024-04-13 17:35:50,385] A new study created in memory with name: no-name-df6360cb-1d3f-4d55-a0bb-8392e95d614f


Fold 5 C-index: 0.630901287553648
[I 2024-04-13 17:35:50,371] Trial 0 finished with value: 0.6377041251337131 and parameters: {}. Best is trial 0 with value: 0.6377041251337131.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6377041251337131], datetime_start=datetime.datetime(2024, 4, 13, 17, 35, 37, 608765), datetime_complete=datetime.datetime(2024, 4, 13, 17, 35, 50, 371165), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6377041251337131


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.2400498117827728
Fold 2 IBS: 0.25100737618728725
Fold 3 IBS: 0.17791096375880883
Fold 4 IBS: 0.2621949716521241
Fold 5 IBS: 0.20430826926374088
[I 2024-04-13 17:35:51,624] Trial 0 finished with value: 0.22709427852894676 and parameters: {}. Best is trial 0 with value: 0.22709427852894676.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.22709427852894676], datetime_start=datetime.datetime(2024, 4, 13, 17, 35, 50, 427960), datetime_complete=datetime.datetime(2024, 4, 13, 17, 35, 51, 621400), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.22709427852894676


In [34]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [35]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.638
train_ibs:  0.227


#### Test

In [36]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [37]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.566
IBS score: 0.262


In [38]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [39]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis

#### Train

In [40]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold( n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 17:35:52,166] A new study created in memory with name: no-name-b8e0c031-8ab5-43dc-8ace-6914b063e9e8


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6334661354581673
Fold 2 C-index: 0.5058139534883721
Fold 3 C-index: 0.5617021276595745


[I 2024-04-13 17:35:52,778] A new study created in memory with name: no-name-eefa01eb-afa3-49a6-869a-e30f083d63c8


Fold 4 C-index: 0.5703422053231939
Fold 5 C-index: 0.6909871244635193
[I 2024-04-13 17:35:52,766] Trial 0 finished with value: 0.5924623092785654 and parameters: {}. Best is trial 0 with value: 0.5924623092785654.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.5924623092785654], datetime_start=datetime.datetime(2024, 4, 13, 17, 35, 52, 199145), datetime_complete=datetime.datetime(2024, 4, 13, 17, 35, 52, 765804), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.5924623092785654


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.24724709674441997
Fold 2 IBS: 0.23203988447801246
Fold 3 IBS: 0.2289818653708319
Fold 4 IBS: 0.24197477208488385
Fold 5 IBS: 0.22939558600665344
[I 2024-04-13 17:35:53,352] Trial 0 finished with value: 0.2359278409369603 and parameters: {}. Best is trial 0 with value: 0.2359278409369603.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.2359278409369603], datetime_start=datetime.datetime(2024, 4, 13, 17, 35, 52, 812056), datetime_complete=datetime.datetime(2024, 4, 13, 17, 35, 53, 351748), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.2359278409369603


In [41]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [42]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.592
train_ibs:  0.236


#### Test

In [43]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [44]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.533


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.229


In [45]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [46]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold( n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 17:35:53,654] A new study created in memory with name: no-name-47bef6a1-04b0-4de9-84a4-bf4312bf7d38


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6055776892430279
Fold 2 C-index: 0.5310077519379846
Fold 3 C-index: 0.7446808510638298
Fold 4 C-index: 0.6806083650190115


[I 2024-04-13 17:35:54,980] A new study created in memory with name: no-name-ddd554fa-54e0-43a8-b1a7-1d0194d54633


Fold 5 C-index: 0.6223175965665236
[I 2024-04-13 17:35:54,971] Trial 0 finished with value: 0.6368384507660754 and parameters: {}. Best is trial 0 with value: 0.6368384507660754.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6368384507660754], datetime_start=datetime.datetime(2024, 4, 13, 17, 35, 53, 699911), datetime_complete=datetime.datetime(2024, 4, 13, 17, 35, 54, 970577), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6368384507660754


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.23952972737617964
Fold 2 IBS: 0.24887322452368552
Fold 3 IBS: 0.17930689607847744
Fold 4 IBS: 0.26095544845313173
Fold 5 IBS: 0.20425265437196208
[I 2024-04-13 17:35:56,962] Trial 0 finished with value: 0.22658359016068727 and parameters: {}. Best is trial 0 with value: 0.22658359016068727.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.22658359016068727], datetime_start=datetime.datetime(2024, 4, 13, 17, 35, 55, 350086), datetime_complete=datetime.datetime(2024, 4, 13, 17, 35, 56, 961201), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.22658359016068727


In [47]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [48]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.637
train_ibs:  0.227


#### Test

In [49]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [50]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.567


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.259


In [51]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [52]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold( n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 17:35:57,919] A new study created in memory with name: no-name-127cc9e9-89e3-455e-9a12-115c6d4dbace


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.601593625498008
Fold 2 C-index: 0.5348837209302325
Fold 3 C-index: 0.7446808510638298
Fold 4 C-index: 0.6806083650190115
Fold 5 C-index: 0.6266094420600858
[I 2024-04-13 17:35:58,940] Trial 0 finished with value: 0.6376752009142335 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.6376752009142335.
Fold 1 C-index: 0.5896414342629482
Fold 2 C-index: 0.5348837209302325
Fold 3 C-index: 0.7446808510638298
Fold 4 C-index: 0.6806083650190115
Fold 5 C-index: 0.6351931330472103
[I 2024-04-13 17:35:59,962] Trial 1 finished with value: 0.6370015008646465 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.6376752009142335.
Fold 1 C-index: 0.5936254980079682
Fold 2 C-index: 0.49806201550387597
Fold 3 C-index: 0.7446808510638298
Fold 4 C-index: 0.6806083650190115
Fold 5 C-index: 0.6351931330472103
[I 2024-04-13 17:36:00,860] Trial 2 finished with value: 0.6304339725283791 and parameters: {'l1_ratio': 0.22692876841884668}.

[I 2024-04-13 17:36:24,921] Trial 23 finished with value: 0.6393919391116584 and parameters: {'l1_ratio': 0.651954326927192}. Best is trial 23 with value: 0.6393919391116584.
Fold 1 C-index: 0.601593625498008
Fold 2 C-index: 0.5310077519379846
Fold 3 C-index: 0.7446808510638298
Fold 4 C-index: 0.6806083650190115
Fold 5 C-index: 0.630901287553648
[I 2024-04-13 17:36:26,199] Trial 24 finished with value: 0.6377583762144964 and parameters: {'l1_ratio': 0.7687494836451149}. Best is trial 23 with value: 0.6393919391116584.
Fold 1 C-index: 0.601593625498008
Fold 2 C-index: 0.5310077519379846
Fold 3 C-index: 0.7446808510638298
Fold 4 C-index: 0.6806083650190115
Fold 5 C-index: 0.6351931330472103
[I 2024-04-13 17:36:28,144] Trial 25 finished with value: 0.6386167453132088 and parameters: {'l1_ratio': 0.6633833508070742}. Best is trial 23 with value: 0.6393919391116584.
Fold 1 C-index: 0.6055776892430279
Fold 2 C-index: 0.5310077519379846
Fold 3 C-index: 0.7446808510638298
Fold 4 C-index: 0.680

Fold 2 C-index: 0.5348837209302325
Fold 3 C-index: 0.7446808510638298
Fold 4 C-index: 0.6806083650190115
Fold 5 C-index: 0.6351931330472103
[I 2024-04-13 17:36:52,420] Trial 47 finished with value: 0.6377983136136505 and parameters: {'l1_ratio': 0.4461628541794258}. Best is trial 23 with value: 0.6393919391116584.
Fold 1 C-index: 0.601593625498008
Fold 2 C-index: 0.5310077519379846
Fold 3 C-index: 0.7446808510638298
Fold 4 C-index: 0.6806083650190115
Fold 5 C-index: 0.6223175965665236
[I 2024-04-13 17:36:53,463] Trial 48 finished with value: 0.6360416380170715 and parameters: {'l1_ratio': 0.7768768486379835}. Best is trial 23 with value: 0.6393919391116584.
Fold 1 C-index: 0.601593625498008
Fold 2 C-index: 0.5348837209302325
Fold 3 C-index: 0.7446808510638298
Fold 4 C-index: 0.6806083650190115
Fold 5 C-index: 0.6351931330472103
[I 2024-04-13 17:36:54,431] Trial 49 finished with value: 0.6393919391116584 and parameters: {'l1_ratio': 0.6350137484940073}. Best is trial 23 with value: 0.63

...

  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 IBS: 0.2399316357515028
Fold 2 IBS: 0.24869689058088437
Fold 3 IBS: 0.17922908696999248
Fold 4 IBS: 0.26102159508422307
Fold 5 IBS: 0.20415166361804699
[I 2024-04-13 17:37:39,657] Trial 0 finished with value: 0.22660617440092995 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.22660617440092995.
Fold 1 IBS: 0.24065183691727052
Fold 2 IBS: 0.24875197100872162
Fold 3 IBS: 0.1792132921209873
Fold 4 IBS: 0.26110143013070647
Fold 5 IBS: 0.20399045974317612
[I 2024-04-13 17:37:40,769] Trial 1 finished with value: 0.22674179798417238 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.22660617440092995.
Fold 1 IBS: 0.240768303616732
Fold 2 IBS: 0.2325812298093743
Fold 3 IBS: 0.17916710097142027
Fold 4 IBS: 0.26109267593358537
Fold 5 IBS: 0.20395622685142237
[I 2024-04-13 17:37:41,620] Trial 2 finished with value: 0.22351310743650687 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 2 with value: 0.22351310743650

Fold 1 IBS: 0.24015153650452095
Fold 2 IBS: 0.24882041004327032
Fold 3 IBS: 0.17922565256396153
Fold 4 IBS: 0.2610612525070375
Fold 5 IBS: 0.20410709163785418
[I 2024-04-13 17:38:01,482] Trial 25 finished with value: 0.2266731886513289 and parameters: {'l1_ratio': 0.5634871043246708}. Best is trial 19 with value: 0.22350091365166636.
Fold 1 IBS: 0.24056048373465547
Fold 2 IBS: 0.24868366867195896
Fold 3 IBS: 0.1792199757947934
Fold 4 IBS: 0.2611287203583251
Fold 5 IBS: 0.20401989073831767
[I 2024-04-13 17:38:02,546] Trial 26 finished with value: 0.22672254785961013 and parameters: {'l1_ratio': 0.3312072877416866}. Best is trial 19 with value: 0.22350091365166636.
Fold 1 IBS: 0.2448965928225129
Fold 2 IBS: 0.2321793735856304
Fold 3 IBS: 0.2271073213455378
Fold 4 IBS: 0.24271598902448063
Fold 5 IBS: 0.22495295734399848
[I 2024-04-13 17:38:02,989] Trial 27 finished with value: 0.23437044682443203 and parameters: {'l1_ratio': 0.09486155085202524}. Best is trial 19 with value: 0.22350091365

Fold 1 IBS: 0.24544072408015
Fold 2 IBS: 0.23210687205360228
Fold 3 IBS: 0.22757503093079218
Fold 4 IBS: 0.24247202223273837
Fold 5 IBS: 0.22599013520701253
[I 2024-04-13 17:38:25,878] Trial 50 finished with value: 0.2347169569008591 and parameters: {'l1_ratio': 0.065098722293608}. Best is trial 19 with value: 0.22350091365166636.
Fold 1 IBS: 0.2407294014440355
Fold 2 IBS: 0.23265072945724835
Fold 3 IBS: 0.17920540061813206
Fold 4 IBS: 0.26107913266507865
Fold 5 IBS: 0.2039928030830184
[I 2024-04-13 17:38:26,752] Trial 51 finished with value: 0.2235314934535026 and parameters: {'l1_ratio': 0.24947029881331234}. Best is trial 19 with value: 0.22350091365166636.
Fold 1 IBS: 0.2408801241466323
Fold 2 IBS: 0.23242139750253155
Fold 3 IBS: 0.17915648712745033
Fold 4 IBS: 0.2611508438114237
Fold 5 IBS: 0.20394448468930906
[I 2024-04-13 17:38:27,829] Trial 52 finished with value: 0.22351066745546938 and parameters: {'l1_ratio': 0.17593565645109632}. Best is trial 19 with value: 0.2235009136516

Fold 1 IBS: 0.2408832088972797
Fold 2 IBS: 0.23238956578423497
Fold 3 IBS: 0.17920977092986604
Fold 4 IBS: 0.2610963777414903
Fold 5 IBS: 0.2039525027840345
[I 2024-04-13 17:38:48,116] Trial 75 finished with value: 0.2235062852273811 and parameters: {'l1_ratio': 0.16577591058395885}. Best is trial 69 with value: 0.2235006290348167.
Fold 1 IBS: 0.24094936744845338
Fold 2 IBS: 0.23231707665139043
Fold 3 IBS: 0.22651229905598805
Fold 4 IBS: 0.2430891087275657
Fold 5 IBS: 0.2236810348665296
[I 2024-04-13 17:38:48,703] Trial 76 finished with value: 0.23330977734998543 and parameters: {'l1_ratio': 0.1423642477817916}. Best is trial 69 with value: 0.2235006290348167.
Fold 1 IBS: 0.24511885091084448
Fold 2 IBS: 0.23214576160364095
Fold 3 IBS: 0.22730154999830518
Fold 4 IBS: 0.24260964197293514
Fold 5 IBS: 0.22537910377810255
[I 2024-04-13 17:38:49,092] Trial 77 finished with value: 0.23451098165276565 and parameters: {'l1_ratio': 0.08183241515980826}. Best is trial 69 with value: 0.22350062903

In [53]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [54]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.639
train_ibs:  0.224


#### Test

In [55]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [56]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.651954326927192)

test_cindex : 0.567


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.16755699923996967)

test_ibs:  0.229


In [57]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [58]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold( n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-13 17:39:10,465] A new study created in memory with name: no-name-c9d4914a-99f4-4ed9-a311-b888ad08b3f4


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5976095617529881
Fold 2 C-index: 0.6143410852713178
Fold 3 C-index: 0.6212765957446809
Fold 4 C-index: 0.6806083650190115
Fold 5 C-index: 0.575107296137339
[I 2024-04-13 17:39:18,418] Trial 0 finished with value: 0.6177885807850675 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.6177885807850675.
Fold 1 C-index: 0.6095617529880478
Fold 2 C-index: 0.624031007751938
Fold 3 C-index: 0.6595744680851063
Fold 4 C-index: 0.7053231939163498
Fold 5 C-index: 0.6030042918454935
[I 2024-04-13 17:39:24,060] Trial 1 finished with value: 0.6402989429173871 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 'm

Fold 3 C-index: 0.7595744680851064
Fold 4 C-index: 0.7053231939163498
Fold 5 C-index: 0.6351931330472103
[I 2024-04-13 17:40:29,226] Trial 15 finished with value: 0.6757811535432272 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 2, 'min_samples_leaf': 2, 'max_depth': 7, 'n_estimators': 120, 'oob_score': True, 'max_samples': 0.9847369290686088, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.17574628605071868, 'warm_start': True}. Best is trial 14 with value: 0.7002369632470599.
Fold 1 C-index: 0.6235059760956175
Fold 2 C-index: 0.6686046511627907
Fold 3 C-index: 0.7212765957446808
Fold 4 C-index: 0.7243346007604563
Fold 5 C-index: 0.648068669527897
[I 2024-04-13 17:40:31,397] Trial 16 finished with value: 0.6771580986582885 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 7, 'min_samples_leaf': 17, 'max_depth': 7, 'n_estimators': 137, 'oob_score': True, 'max_samples': 0.962365055520953, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.1903778147805238

Fold 1 C-index: 0.6215139442231076
Fold 2 C-index: 0.6744186046511628
Fold 3 C-index: 0.774468085106383
Fold 4 C-index: 0.7395437262357415
Fold 5 C-index: 0.6952789699570815
[I 2024-04-13 17:40:58,272] Trial 30 finished with value: 0.7010446660346952 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 17, 'min_samples_leaf': 8, 'max_depth': 14, 'n_estimators': 169, 'oob_score': True, 'max_samples': 0.5345414788093061, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.03755841239520112, 'warm_start': True}. Best is trial 29 with value: 0.7116053815973882.
Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.6666666666666666
Fold 3 C-index: 0.7787234042553192
Fold 4 C-index: 0.7357414448669202
Fold 5 C-index: 0.7017167381974249
[I 2024-04-13 17:41:00,832] Trial 31 finished with value: 0.7000756268928836 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 17, 'min_samples_leaf': 8, 'max_depth': 14, 'n_estimators': 171, 'oob_score': True, 'max_samples': 0.5019192620095752

...

Fold 5 IBS: 0.2108108318582224
[I 2024-04-13 17:51:40,922] Trial 31 finished with value: 0.2206877222460263 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 9, 'min_samples_leaf': 9, 'max_depth': 3, 'n_estimators': 396, 'oob_score': True, 'max_samples': 0.2868712535426494, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.021231453689495153}. Best is trial 26 with value: 0.22015707552217512.
Fold 1 IBS: 0.24659836091991297
Fold 2 IBS: 0.23212875696426277
Fold 3 IBS: 0.23013831975076382
Fold 4 IBS: 0.24109773536375373
Fold 5 IBS: 0.22997643124737513
[I 2024-04-13 17:51:53,573] Trial 32 finished with value: 0.23598792084921366 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 10, 'min_samples_leaf': 9, 'max_depth': 3, 'n_estimators': 323, 'oob_score': True, 'max_samples': 0.15457524910822473, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.004932622259272009}. Best is trial 26 with value: 0.22015707552217512.
Fold 1 IBS: 0.23197141083929648
Fold 2 IBS: 0.22

Fold 1 IBS: 0.23103878287252402
Fold 2 IBS: 0.22628918231285983
Fold 3 IBS: 0.2133979486579942
Fold 4 IBS: 0.22430579562693312
Fold 5 IBS: 0.21153361401748674
[I 2024-04-13 17:56:10,149] Trial 47 finished with value: 0.22131306469755957 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 8, 'min_samples_leaf': 11, 'max_depth': 10, 'n_estimators': 475, 'oob_score': False, 'max_samples': 0.43491671167283275, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.07826676341691717}. Best is trial 26 with value: 0.22015707552217512.
Fold 1 IBS: 0.2461442212529148
Fold 2 IBS: 0.23226781881803804
Fold 3 IBS: 0.22981312479103597
Fold 4 IBS: 0.24139414747597257
Fold 5 IBS: 0.2304782442201671
[I 2024-04-13 17:56:28,307] Trial 48 finished with value: 0.23601951131162568 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 3, 'min_samples_leaf': 19, 'max_depth': 6, 'n_estimators': 447, 'oob_score': False, 'max_samples': 0.2751094610735481, 'max_features': 'auto', 'min_weight_fraction_

Fold 5 IBS: 0.23018854152461504
[I 2024-04-13 18:00:13,196] Trial 62 finished with value: 0.23591814060376043 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 5, 'min_samples_leaf': 5, 'max_depth': 10, 'n_estimators': 455, 'oob_score': True, 'max_samples': 0.4677262455345284, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.2827349986373208}. Best is trial 26 with value: 0.22015707552217512.
Fold 1 IBS: 0.24652405780792855
Fold 2 IBS: 0.2322152403880562
Fold 3 IBS: 0.22946718005561817
Fold 4 IBS: 0.2414651127800749
Fold 5 IBS: 0.23031058287471998
[I 2024-04-13 18:00:46,151] Trial 63 finished with value: 0.23599643478127957 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 8, 'min_samples_leaf': 14, 'max_depth': 9, 'n_estimators': 499, 'oob_score': True, 'max_samples': 0.36524236153301975, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.19147391488941964}. Best is trial 26 with value: 0.22015707552217512.
Fold 1 IBS: 0.23373423023692633
Fold 2 IBS: 0.2252

Fold 1 IBS: 0.24217247193439537
Fold 2 IBS: 0.2278793635566358
Fold 3 IBS: 0.20080078124083145
Fold 4 IBS: 0.22310960096298033
Fold 5 IBS: 0.2079672399605948
[I 2024-04-13 18:04:51,792] Trial 78 finished with value: 0.22038589153108754 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 4, 'min_samples_leaf': 2, 'max_depth': 13, 'n_estimators': 418, 'oob_score': False, 'max_samples': 0.7742508336660989, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.033857990321958055}. Best is trial 71 with value: 0.21814006458981802.
Fold 1 IBS: 0.2355555064593619
Fold 2 IBS: 0.22224942419207222
Fold 3 IBS: 0.20877935998830383
Fold 4 IBS: 0.22444536984656233
Fold 5 IBS: 0.2081228365598182
[I 2024-04-13 18:05:05,919] Trial 79 finished with value: 0.21983049940922367 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 2, 'min_samples_leaf': 4, 'max_depth': 14, 'n_estimators': 439, 'oob_score': False, 'max_samples': 0.8561349620215079, 'max_features': 'log2', 'min_weight_fraction_le

Fold 5 IBS: 0.2069805372099246
[I 2024-04-13 18:08:45,488] Trial 93 finished with value: 0.21836991119421248 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 3, 'min_samples_leaf': 3, 'max_depth': 15, 'n_estimators': 436, 'oob_score': False, 'max_samples': 0.7876680525303928, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.02818336790921719}. Best is trial 71 with value: 0.21814006458981802.
Fold 1 IBS: 0.2481959844323671
Fold 2 IBS: 0.23173657169109757
Fold 3 IBS: 0.20201510994966232
Fold 4 IBS: 0.22255626189257152
Fold 5 IBS: 0.21061654425734486
[I 2024-04-13 18:09:03,380] Trial 94 finished with value: 0.22302409444460863 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 5, 'min_samples_leaf': 3, 'max_depth': 16, 'n_estimators': 436, 'oob_score': False, 'max_samples': 0.8362507108284499, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.027504161893380293}. Best is trial 71 with value: 0.21814006458981802.
Fold 1 IBS: 0.23915650996574608
Fold 2 IBS: 0.223

In [59]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [60]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.79
train_ibs:  0.218


#### Test

In [61]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [62]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=20, max_features='auto', max_leaf_nodes=12,
                     max_samples=0.9937352645069003, min_samples_leaf=1,
                     min_samples_split=5,
                     min_weight_fraction_leaf=0.014733302673387358,
                     n_estimators=435, random_state=123, warm_start=True)

test_cindex:  0.603


RandomSurvivalForest(max_depth=13, max_features='log2', max_leaf_nodes=3,
                     max_samples=0.785271119394671, min_samples_leaf=4,
                     min_samples_split=2,
                     min_weight_fraction_leaf=8.074256882937306e-05,
                     n_estimators=444, random_state=123)

test_ibs:  0.215


In [63]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [64]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [65]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold( n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 18:10:31,065] A new study created in memory with name: no-name-ad38897a-9a17-4961-9c85-ce24e608d10a


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5976095617529881
Fold 2 C-index: 0.6666666666666666
Fold 3 C-index: 0.7382978723404255
Fold 4 C-index: 0.7319391634980988
Fold 5 C-index: 0.6909871244635193
[I 2024-04-13 18:10:33,939] Trial 0 finished with value: 0.6851000777443397 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.6851000777443397.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 18:10:41,899] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531

Fold 2 C-index: 0.6550387596899225
Fold 3 C-index: 0.7212765957446808
Fold 4 C-index: 0.7243346007604563
Fold 5 C-index: 0.6158798283261803
[I 2024-04-13 18:12:01,427] Trial 15 finished with value: 0.6707959967448854 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 9, 'min_samples_leaf': 13, 'max_depth': 4, 'n_estimators': 258, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.6880212408103346, 'min_weight_fraction_leaf': 0.07884420972256234}. Best is trial 12 with value: 0.6881534529891422.
Fold 1 C-index: 0.6055776892430279
Fold 2 C-index: 0.6589147286821705
Fold 3 C-index: 0.7085106382978723
Fold 4 C-index: 0.6977186311787072
Fold 5 C-index: 0.6094420600858369
[I 2024-04-13 18:12:05,372] Trial 16 finished with value: 0.656032749497523 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.8750

Fold 1 C-index: 0.6215139442231076
Fold 2 C-index: 0.6763565891472868
Fold 3 C-index: 0.7446808510638298
Fold 4 C-index: 0.7395437262357415
Fold 5 C-index: 0.628755364806867
[I 2024-04-13 18:13:03,448] Trial 30 finished with value: 0.6821700950953665 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 3, 'min_samples_leaf': 4, 'max_depth': 12, 'n_estimators': 355, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7262614295600427, 'min_weight_fraction_leaf': 0.020774312173092817}. Best is trial 23 with value: 0.6906324467750945.
Fold 1 C-index: 0.6095617529880478
Fold 2 C-index: 0.6550387596899225
Fold 3 C-index: 0.7404255319148936
Fold 4 C-index: 0.7186311787072244
Fold 5 C-index: 0.7017167381974249
[I 2024-04-13 18:13:06,756] Trial 31 finished with value: 0.6850747922995026 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 5, 'min_samples_leaf': 1, 'max_depth': 5, 'n_estimators': 400, 'oob_score': False, 'warm_start': True, 'max_features

...

Fold 1 IBS: 0.23738806550248728
Fold 2 IBS: 0.21993578889192347
Fold 3 IBS: 0.21089498091706652
Fold 4 IBS: 0.2272048730326933
Fold 5 IBS: 0.2132773803587104
[I 2024-04-13 18:23:00,560] Trial 30 finished with value: 0.2217402177405762 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 14, 'min_samples_leaf': 13, 'max_depth': 9, 'n_estimators': 206, 'oob_score': True, 'warm_start': False, 'max_features': 1, 'max_samples': 0.7311100810851326, 'min_weight_fraction_leaf': 0.16969817682654836}. Best is trial 20 with value: 0.21858490307862474.
Fold 1 IBS: 0.2371472983744441
Fold 2 IBS: 0.2192317816045922
Fold 3 IBS: 0.2066273024000569
Fold 4 IBS: 0.2223268600858535
Fold 5 IBS: 0.20805506169911736
[I 2024-04-13 18:23:12,145] Trial 31 finished with value: 0.21867766083281284 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 17, 'min_samples_leaf': 17, 'max_depth': 3, 'n_estimators': 259, 'oob_score': True, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0.84794250

Fold 1 IBS: 0.24647150063770518
Fold 2 IBS: 0.2324005238674857
Fold 3 IBS: 0.2295165680261027
Fold 4 IBS: 0.2413350176086155
Fold 5 IBS: 0.23048981014618952
[I 2024-04-13 18:25:55,751] Trial 45 finished with value: 0.2360426840572197 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 18, 'min_samples_leaf': 18, 'max_depth': 8, 'n_estimators': 184, 'oob_score': True, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0.42889080150309455, 'min_weight_fraction_leaf': 0.30395190085793466}. Best is trial 33 with value: 0.21855367964170153.
Fold 1 IBS: 0.23684554782082465
Fold 2 IBS: 0.21919333404974234
Fold 3 IBS: 0.2053013505980981
Fold 4 IBS: 0.22292477589597334
Fold 5 IBS: 0.20836678612341789
[I 2024-04-13 18:26:08,878] Trial 46 finished with value: 0.21852635889761127 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 20, 'min_samples_leaf': 19, 'max_depth': 6, 'n_estimators': 273, 'oob_score': True, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0

Fold 1 IBS: 0.2427874690307322
Fold 2 IBS: 0.22051307489638503
Fold 3 IBS: 0.21103171261194614
Fold 4 IBS: 0.22387533549578809
Fold 5 IBS: 0.2083271491005745
[I 2024-04-13 18:29:22,448] Trial 60 finished with value: 0.22130694822708522 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 19, 'min_samples_leaf': 1, 'max_depth': 4, 'n_estimators': 74, 'oob_score': False, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0.7017596527420207, 'min_weight_fraction_leaf': 0.09966382421080089}. Best is trial 46 with value: 0.21852635889761127.
Fold 1 IBS: 0.23644800149579903
Fold 2 IBS: 0.2195369287078321
Fold 3 IBS: 0.20653996690015303
Fold 4 IBS: 0.22236489358251474
Fold 5 IBS: 0.20806793260763023
[I 2024-04-13 18:29:35,128] Trial 61 finished with value: 0.21859154465878583 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 18, 'max_depth': 1, 'n_estimators': 357, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 

Fold 1 IBS: 0.23757083460058645
Fold 2 IBS: 0.21936835681341144
Fold 3 IBS: 0.2050586023900099
Fold 4 IBS: 0.2231460133461373
Fold 5 IBS: 0.20844072705559955
[I 2024-04-13 18:32:56,062] Trial 75 finished with value: 0.21871690684114892 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 20, 'min_samples_leaf': 19, 'max_depth': 2, 'n_estimators': 220, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 0.7803578114864529, 'min_weight_fraction_leaf': 0.017478919548903803}. Best is trial 73 with value: 0.21838002210423152.
Fold 1 IBS: 0.23640141704565878
Fold 2 IBS: 0.2197885256064894
Fold 3 IBS: 0.20318516882698215
Fold 4 IBS: 0.22544681939994865
Fold 5 IBS: 0.20897452087659646
[I 2024-04-13 18:33:08,302] Trial 76 finished with value: 0.21875929035113512 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 20, 'min_samples_leaf': 20, 'max_depth': 3, 'n_estimators': 293, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples'

Fold 1 IBS: 0.237217800059635
Fold 2 IBS: 0.219560418627486
Fold 3 IBS: 0.2055584995526966
Fold 4 IBS: 0.22351950054338868
Fold 5 IBS: 0.20798078328178596
[I 2024-04-13 18:35:29,118] Trial 90 finished with value: 0.21876740041299841 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 19, 'min_samples_leaf': 11, 'max_depth': 5, 'n_estimators': 306, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.6578960771293777, 'min_weight_fraction_leaf': 0.16477439734294264}. Best is trial 73 with value: 0.21838002210423152.
Fold 1 IBS: 0.236951540585596
Fold 2 IBS: 0.21950546902732596
Fold 3 IBS: 0.20504430850227562
Fold 4 IBS: 0.22315671121676062
Fold 5 IBS: 0.20782359042139878
[I 2024-04-13 18:35:36,215] Trial 91 finished with value: 0.21849632395067142 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 19, 'min_samples_leaf': 8, 'max_depth': 6, 'n_estimators': 282, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.69

In [66]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [67]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.748
train_ibs:  0.218


#### Test

In [68]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [69]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=11, max_leaf_nodes=17,
                   max_samples=0.8662742964997368, min_samples_leaf=1,
                   min_samples_split=19,
                   min_weight_fraction_leaf=0.0017689713740798967,
                   n_estimators=54, oob_score=True, random_state=123,
                   warm_start=True)

C-index score: 0.585


ExtraSurvivalTrees(max_depth=2, max_features='auto', max_leaf_nodes=20,
                   max_samples=0.7842377375661996, min_samples_leaf=19,
                   min_samples_split=19,
                   min_weight_fraction_leaf=0.04843949872109438,
                   n_estimators=291, oob_score=True, random_state=123)

IBS: 0.213


In [70]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [71]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold( n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-13 18:36:48,867] A new study created in memory with name: no-name-b681894a-23c4-4b5d-acc5-5368af7afc90


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 18:37:43,345] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 18:38:16,602] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 18:53:32,518] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 12 with value: 0.6288581469840132.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 18:55:23,983] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'squar

Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 19:17:06,504] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.7565257046780437, 'learning_rate': 0.01188344684416992, 'dropout_rate': 0.3433523077110169, 'n_estimators': 318, 'criterion': 'squared_error', 'ccp_alpha': 2.063559212730723, 'min_weight_fraction_leaf': 0.25401273903421573, 'max_features': 'auto', 'min_impurity_decrease': 5.773435946665558e-07, 'validation_fraction': 0.8062268184400477, 'min_samples_split': 16, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 3}. Best is trial 12 with value: 0.6288581469840132.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 19:20:11,506] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.8922404482621683, 'learning_rate': 0.011585260292674536, 'dropout_rate': 0.2527000999648632, 'n_estimators': 446, 'criterion': 'friedman_mse', 'ccp_alpha': 0.11820935501930148, 'min_weight_fraction

...

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 21:30:41,914] Trial 55 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.804828042321909, 'learning_rate': 0.023646082998228058, 'dropout_rate': 0.23926982708415356, 'n_estimators': 162, 'criterion': 'squared_error', 'ccp_alpha': 0.40074886285106287, 'min_weight_fraction_leaf': 0.2779066644542128, 'max_features': 'sqrt', 'min_impurity_decrease': 0.008146563872249941, 'validation_fraction': 0.8868940629916056, 'min_samples_split': 4, 'max_leaf_nodes': 16, 'min_samples_leaf': 19, 'max_depth': 4}. Best is trial 45 with value: 0.2332431097060344.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 21:31:04,906] Trial 56 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.6106355356441384, 'learning_rate': 0.0175381503002972, 'dropout_rate': 0.184039934

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 21:33:08,123] Trial 66 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9043457557527063, 'learning_rate': 0.08720647345857328, 'dropout_rate': 0.27193939046019544, 'n_estimators': 154, 'criterion': 'squared_error', 'ccp_alpha': 1.4111498627316026, 'min_weight_fraction_leaf': 0.23517339076530247, 'max_features': 'sqrt', 'min_impurity_decrease': 6.086951842225704e-07, 'validation_fraction': 0.9127218528145804, 'min_samples_split': 4, 'max_leaf_nodes': 15, 'min_samples_leaf': 20, 'max_depth': 2}. Best is trial 63 with value: 0.23223812135106253.
Fold 1 IBS: 0.24724710044658996
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 21:33:10,017] Trial 67 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.8250695857763196, 'learning_rate': 0.0911086342832341, 'dropout_rate': 0.3723000

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 21:35:24,390] Trial 77 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.7428517276581355, 'learning_rate': 0.07706989744762442, 'dropout_rate': 0.4476209058480567, 'n_estimators': 91, 'criterion': 'squared_error', 'ccp_alpha': 1.5888723787914292, 'min_weight_fraction_leaf': 0.16206442611838873, 'max_features': 0.1, 'min_impurity_decrease': 3.33671674086432e-07, 'validation_fraction': 0.9402673106586569, 'min_samples_split': 12, 'max_leaf_nodes': 13, 'min_samples_leaf': 17, 'max_depth': 9}. Best is trial 63 with value: 0.23223812135106253.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 21:35:34,532] Trial 78 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.7715268987373948, 'learning_rate': 0.08985503754923349, 'dropout_rate': 0.51064243866

Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 21:36:53,058] Trial 88 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.6413430388178152, 'learning_rate': 0.0973528090376961, 'dropout_rate': 0.26912242502602596, 'n_estimators': 174, 'criterion': 'squared_error', 'ccp_alpha': 0.5807061908097002, 'min_weight_fraction_leaf': 0.16723537370276506, 'max_features': 0.1, 'min_impurity_decrease': 1.190634063232502e-06, 'validation_fraction': 0.8005899065401637, 'min_samples_split': 9, 'max_leaf_nodes': 11, 'min_samples_leaf': 17, 'max_depth': 4}. Best is trial 82 with value: 0.23205660370863482.
Fold 1 IBS: 0.24220714862378384
Fold 2 IBS: 0.22705284477038132
Fold 3 IBS: 0.22374120154251026
Fold 4 IBS: 0.23761138561741843
Fold 5 IBS: 0.22312495335199894
[I 2024-04-13 21:37:06,408] Trial 89 finished with value: 0.23074750678121858 and parameters: {'subsample': 0.38730242337646104, 'learning_rate': 0.08109800836387941, 'dropout_rate': 0.18122999

Fold 4 IBS: 0.23852342496708018
Fold 5 IBS: 0.22486084489133845
[I 2024-04-13 21:39:40,076] Trial 99 finished with value: 0.23142156664291713 and parameters: {'subsample': 0.40791967441342253, 'learning_rate': 0.07712333235501795, 'dropout_rate': 0.19859974199600133, 'n_estimators': 143, 'criterion': 'squared_error', 'ccp_alpha': 0.004426170739568851, 'min_weight_fraction_leaf': 0.16314282359709542, 'max_features': 'log2', 'min_impurity_decrease': 5.353566353972388e-07, 'validation_fraction': 0.1604355124738367, 'min_samples_split': 11, 'max_leaf_nodes': 15, 'min_samples_leaf': 14, 'max_depth': 4}. Best is trial 91 with value: 0.22930168836973525.


* Best trial for IBS: 
 FrozenTrial(number=91, state=TrialState.COMPLETE, values=[0.22930168836973525], datetime_start=datetime.datetime(2024, 4, 13, 21, 37, 33, 748497), datetime_complete=datetime.datetime(2024, 4, 13, 21, 37, 48, 859878), params={'subsample': 0.3646963250854607, 'learning_rate': 0.09255241886126767, 'dropout_rate': 0.1236

In [72]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [73]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.648
train_ibs:  0.229


#### Test

In [74]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [75]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.02647951541473475,
                                 criterion='squared_error',
                                 dropout_rate=0.14303111133639826,
                                 learning_rate=0.05317196534413593,
                                 max_depth=19, max_features=0.1,
                                 max_leaf_nodes=15,
                                 min_impurity_decrease=7.356647091617949e-06,
                                 min_samples_leaf=17, min_samples_split=19,
                                 min_weight_fraction_leaf=0.35066087197674667,
                                 n_estimators=78, random_state=123,
                                 subsample=0.9736544653699906,
                                 validation_fraction=0.6117331205297228)

C-index score: 0.639


GradientBoostingSurvivalAnalysis(ccp_alpha=0.03832008885345805,
                                 criterion='squared_error',
                                 dropout_rate=0.12363276406693596,
                                 learning_rate=0.09255241886126767, max_depth=4,
                                 max_leaf_nodes=15,
                                 min_impurity_decrease=6.12703415921239e-07,
                                 min_samples_leaf=16, min_samples_split=13,
                                 min_weight_fraction_leaf=0.14198218649980035,
                                 n_estimators=143, random_state=123,
                                 subsample=0.3646963250854607,
                                 validation_fraction=0.965806971606102)

IBS: 0.221


In [76]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [77]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [78]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold( n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 21:39:44,118] A new study created in memory with name: no-name-ccd8a36d-73e0-4c21-b396-77d556801cfe


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.599601593625498
Fold 2 C-index: 0.5096899224806202
Fold 3 C-index: 0.5553191489361702
Fold 4 C-index: 0.6482889733840305
Fold 5 C-index: 0.6201716738197425
[I 2024-04-13 21:39:45,847] Trial 0 finished with value: 0.5866142624492122 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.5866142624492122.
Fold 1 C-index: 0.599601593625498
Fold 2 C-index: 0.5096899224806202
Fold 3 C-index: 0.5553191489361702
Fold 4 C-index: 0.5931558935361216
Fold 5 C-index: 0.6244635193133047
[I 2024-04-13 21:39:59,513] Trial 1 finished with value: 0.576446015578343 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.5866142624492122.
Fold 1 C-index: 0.599601593625498
Fold 2 C-index: 0.5096899224806202
Fold 3 C-index: 0.5553191489361702
Fold 4 C-

Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.5310077519379846
Fold 3 C-index: 0.597872340425532
Fold 4 C-index: 0.6045627376425855
Fold 5 C-index: 0.6437768240343348
[I 2024-04-13 21:41:59,173] Trial 19 finished with value: 0.5981530941547009 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.7254192287154788, 'n_estimators': 117, 'learning_rate': 0.09614402133777997}. Best is trial 12 with value: 0.6083561218656349.
Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.5271317829457365
Fold 3 C-index: 0.5808510638297872
Fold 4 C-index: 0.6007604562737643
Fold 5 C-index: 0.648068669527897
[I 2024-04-13 21:42:11,429] Trial 20 finished with value: 0.5940715578620506 and parameters: {'subsample': 0.2630057481431337, 'dropout_rate': 0.18040218016888274, 'n_estimators': 423, 'learning_rate': 0.07788582119853761}. Best is trial 12 with value: 0.6083561218656349.
Fold 1 C-index: 0.6215139442231076
Fold 2 C-index: 0.5310077519379846
Fold 3 C-index: 0.6085106382978723
F

Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.5251937984496124
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6007604562737643
Fold 5 C-index: 0.6523605150214592
[I 2024-04-13 21:44:25,731] Trial 38 finished with value: 0.5907125428274955 and parameters: {'subsample': 0.2765053049871465, 'dropout_rate': 0.49675407246821296, 'n_estimators': 46, 'learning_rate': 0.07843779087354608}. Best is trial 37 with value: 0.63135676757979.
Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.5096899224806202
Fold 3 C-index: 0.5531914893617021
Fold 4 C-index: 0.596958174904943
Fold 5 C-index: 0.648068669527897
[I 2024-04-13 21:44:28,499] Trial 39 finished with value: 0.584290814601646 and parameters: {'subsample': 0.4186552843670305, 'dropout_rate': 0.4122412566560423, 'n_estimators': 159, 'learning_rate': 0.05598015324329953}. Best is trial 37 with value: 0.63135676757979.
Fold 1 C-index: 0.599601593625498
Fold 2 C-index: 0.5096899224806202
Fold 3 C-index: 0.5553191489361702
Fold 4 C-

...

Fold 3 IBS: 0.23443639398619098
Fold 4 IBS: 0.24374854938649268
Fold 5 IBS: 0.20898654378339646
[I 2024-04-13 21:47:21,591] Trial 19 finished with value: 0.2333917640587518 and parameters: {'subsample': 0.3892284838807411, 'dropout_rate': 0.5354432466556798, 'n_estimators': 131, 'learning_rate': 0.023294697221931306}. Best is trial 7 with value: 0.23106060701964246.
Fold 1 IBS: 0.244476360186189
Fold 2 IBS: 0.2305472199436817
Fold 3 IBS: 0.22377140228384404
Fold 4 IBS: 0.2386766254699274
Fold 5 IBS: 0.22296335334244682
[I 2024-04-13 21:47:22,527] Trial 20 finished with value: 0.23208699224521778 and parameters: {'subsample': 0.6210873354088753, 'dropout_rate': 0.7933084651006226, 'n_estimators': 66, 'learning_rate': 0.013007963411749002}. Best is trial 7 with value: 0.23106060701964246.
Fold 1 IBS: 0.24513654608372853
Fold 2 IBS: 0.23094110628575634
Fold 3 IBS: 0.22464827063854145
Fold 4 IBS: 0.23926825072734292
Fold 5 IBS: 0.22475950483168106
[I 2024-04-13 21:47:23,406] Trial 21 finis

Fold 3 IBS: 0.26639509455111465
Fold 4 IBS: 0.2932421765596226
Fold 5 IBS: 0.21878189898179665
[I 2024-04-13 21:47:47,574] Trial 38 finished with value: 0.28437186803105347 and parameters: {'subsample': 0.10133044273581891, 'dropout_rate': 0.29839135236038283, 'n_estimators': 180, 'learning_rate': 0.07288612365177065}. Best is trial 37 with value: 0.22864384564709322.
Fold 1 IBS: 0.24367533970078817
Fold 2 IBS: 0.230581843797866
Fold 3 IBS: 0.22426510493646246
Fold 4 IBS: 0.24086016296070725
Fold 5 IBS: 0.21485542180084868
[I 2024-04-13 21:47:48,346] Trial 39 finished with value: 0.2308475746393345 and parameters: {'subsample': 0.23574838842029588, 'dropout_rate': 0.44092575231554404, 'n_estimators': 31, 'learning_rate': 0.06528568864007542}. Best is trial 37 with value: 0.22864384564709322.
Fold 1 IBS: 0.38495903085339733
Fold 2 IBS: 0.37839237180942703
Fold 3 IBS: 0.31605836309462204
Fold 4 IBS: 0.3964877229235561
Fold 5 IBS: 0.33876619173980255
[I 2024-04-13 21:48:00,309] Trial 40 f

Fold 3 IBS: 0.2967856756039888
Fold 4 IBS: 0.38328944705560486
Fold 5 IBS: 0.3356135313416512
[I 2024-04-13 21:48:27,782] Trial 57 finished with value: 0.3620474670786823 and parameters: {'subsample': 0.1024539420046288, 'dropout_rate': 0.49346965615517374, 'n_estimators': 337, 'learning_rate': 0.08172067337182386}. Best is trial 43 with value: 0.22789865509331322.
Fold 1 IBS: 0.25416538597195026
Fold 2 IBS: 0.25706955072107596
Fold 3 IBS: 0.23764359423215053
Fold 4 IBS: 0.2677073268135179
Fold 5 IBS: 0.1957637741799054
[I 2024-04-13 21:48:29,507] Trial 58 finished with value: 0.24246992638372 and parameters: {'subsample': 0.16049662250408656, 'dropout_rate': 0.2210250116180224, 'n_estimators': 104, 'learning_rate': 0.06332357776860716}. Best is trial 43 with value: 0.22789865509331322.
Fold 1 IBS: 0.24278576940412266
Fold 2 IBS: 0.23400445256009053
Fold 3 IBS: 0.2332137098103916
Fold 4 IBS: 0.24553977005200386
Fold 5 IBS: 0.20644973367875002
[I 2024-04-13 21:48:30,564] Trial 59 finish

Fold 1 IBS: 0.24614517653453177
Fold 2 IBS: 0.23173156309822698
Fold 3 IBS: 0.22767587336332618
Fold 4 IBS: 0.2363604585844503
Fold 5 IBS: 0.21236293144974608
[I 2024-04-13 21:48:57,687] Trial 77 finished with value: 0.2308552006060563 and parameters: {'subsample': 0.7888221964587577, 'dropout_rate': 0.17771784288698328, 'n_estimators': 28, 'learning_rate': 0.08836993630729562}. Best is trial 43 with value: 0.22789865509331322.
Fold 1 IBS: 0.3464250655392121
Fold 2 IBS: 0.37879980708527167
Fold 3 IBS: 0.30975364865243243
Fold 4 IBS: 0.38010482401937984
Fold 5 IBS: 0.3347877068439646
[I 2024-04-13 21:49:02,854] Trial 78 finished with value: 0.34997421042805216 and parameters: {'subsample': 0.2643775590636606, 'dropout_rate': 0.3702237451622687, 'n_estimators': 226, 'learning_rate': 0.09936039441108106}. Best is trial 43 with value: 0.22789865509331322.
Fold 1 IBS: 0.24592889853108682
Fold 2 IBS: 0.2405319338011287
Fold 3 IBS: 0.2395407535809336
Fold 4 IBS: 0.25411025908718043
Fold 5 IBS

Fold 2 IBS: 0.23283374383732752
Fold 3 IBS: 0.214629733795403
Fold 4 IBS: 0.24479707089702957
Fold 5 IBS: 0.20252984019637682
[I 2024-04-13 21:49:21,834] Trial 96 finished with value: 0.22707035076234602 and parameters: {'subsample': 0.10431216939439927, 'dropout_rate': 0.4866966976769669, 'n_estimators': 36, 'learning_rate': 0.08781069522287915}. Best is trial 95 with value: 0.22694193219274764.
Fold 1 IBS: 0.24226623933078084
Fold 2 IBS: 0.23689506425405096
Fold 3 IBS: 0.23668323797670957
Fold 4 IBS: 0.2663621823478399
Fold 5 IBS: 0.19962959656722443
[I 2024-04-13 21:49:22,671] Trial 97 finished with value: 0.2363672640953211 and parameters: {'subsample': 0.10863299078416362, 'dropout_rate': 0.5772422505268303, 'n_estimators': 50, 'learning_rate': 0.08730182830746808}. Best is trial 95 with value: 0.22694193219274764.
Fold 1 IBS: 0.2668207214584027
Fold 2 IBS: 0.29605961251296825
Fold 3 IBS: 0.24890800068026245
Fold 4 IBS: 0.2820742952494582
Fold 5 IBS: 0.2013620297024855
[I 2024-04-

In [79]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [80]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.631
train_ibs:  0.227


#### Test

In [81]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [82]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.3917773398227318,
                                              learning_rate=0.05621431261796065,
                                              n_estimators=1, random_state=123,
                                              subsample=0.14187696914743586)

C-index score: 0.51


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.4952232489349569,
                                              learning_rate=0.08744198973781114,
                                              n_estimators=35, random_state=123,
                                              subsample=0.10055436608530686)

IBS: 0.233


In [83]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

# Results

In [84]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.790,1.0
ExtraSurvivalTrees,0.748,2.0
GradientBoosting,0.648,3.0
CoxElastic,0.639,4.0
CoxPH,0.638,5.0
CoxLasso,0.637,6.0
ComponentwiseGradientBoosting,0.631,7.0
CoxRidge,0.592,8.0


In [85]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
Randomsurvivalforest,0.218,1.5
ExtraSurvivalTrees,0.218,1.5
CoxElastic,0.224,3.0
CoxPH,0.227,5.0
CoxLasso,0.227,5.0
ComponentwiseGradientBoosting,0.227,5.0
GradientBoosting,0.229,7.0
CoxRidge,0.236,8.0


In [86]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
GradientBoosting,0.639,1.0
Randomsurvivalforest,0.603,2.0
ExtraSurvivalTrees,0.585,3.0
CoxLasso,0.567,4.5
CoxElastic,0.567,4.5
CoxPH,0.566,6.0
CoxRidge,0.533,7.0
ComponentwiseGradientBoosting,0.510,8.0


In [87]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs 

,IBS,rank
ExtraSurvivalTrees,0.213,1.0
Randomsurvivalforest,0.215,2.0
GradientBoosting,0.221,3.0
CoxRidge,0.229,4.5
CoxElastic,0.229,4.5
ComponentwiseGradientBoosting,0.233,6.0
CoxLasso,0.259,7.0
CoxPH,0.262,8.0


In [89]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = 'path_to_your_folder/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d1/dfs/yeojohnson/rent/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d1_dfs_yeojohnson_rent_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [90]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-13
